#01 – Carga de datos crudos a BigQuery (staging)

Carga el archivo taxi_trip_data.csv (10.000.000 de registros) en la tabla staging.taxi_trip_data del proyecto dw-taxi-nyc.
Se usa Colab porque el archivo supera el límite de 100 MB de carga directa de la consola de BigQuery.

**Requisitos:** el archivo taxi_trip_data.csv en la raíz de Google Drive (Mi unidad) y acceso al proyecto dw-taxi-nyc.

## 1. Conexión a Google Drive y Google Cloud
Monta Google Drive para leer el CSV y autentica la cuenta con permisos sobre el proyecto de BigQuery.


In [ ]:
from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()

Mounted at /content/drive


## 2. Inspección inicial del archivo
Verifica el tamaño del archivo (1,37 GB) y muestra las primeras filas (fase de extracción del ETL).

In [ ]:
import os
import pandas as pd

RUTA = '/content/drive/MyDrive/taxi_trip_data.csv'
print(f"Tamaño: {os.path.getsize(RUTA) / 1024**3:.2f} GB")
pd.read_csv(RUTA, nrows=5)

Tamaño: 1.37 GB


,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code,store_and_fwd_flag,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,imp_surcharge,total_amount,pickup_location_id,dropoff_location_id
0,2,2018-03-29 13:37:13,2018-03-29 14:17:01,1,18.15,3,N,1,70.0,0.0,0.0,16.16,10.50,0.3,96.96,161,1
1,2,2018-03-29 13:37:18,2018-03-29 14:15:33,1,4.59,1,N,1,25.0,0.0,0.5,5.16,0.00,0.3,30.96,13,230
2,2,2018-03-29 13:26:57,2018-03-29 13:28:03,1,0.30,1,N,1,3.0,0.0,0.5,0.76,0.00,0.3,4.56,231,231
3,2,2018-03-29 13:07:48,2018-03-29 14:03:05,2,16.97,1,N,1,49.5,0.0,0.5,5.61,5.76,0.3,61.67,231,138
4,2,2018-03-29 14:19:11,2018-03-29 15:19:59,5,14.45,1,N,1,45.5,0.0,0.5,10.41,5.76,0.3,62.47,87,138


## 3. Carga a staging
Todas las columnas se cargan como STRING: la capa staging conserva el dato crudo sin transformar. La conversión de tipos se hace después, en la fase de transformación (SQL en BigQuery).

WRITE_TRUNCATE reemplaza la tabla si se vuelve a ejecutar, para evitar duplicados.

Resultado esperado: 10.000.000 de filas.

In [ ]:
from google.cloud import bigquery

PROYECTO = 'dw-taxi-nyc'
TABLA = f'{PROYECTO}.staging.taxi_trip_data'
client = bigquery.Client(project=PROYECTO)

# Todas las columnas como texto: staging guarda el dato crudo
columnas = pd.read_csv(RUTA, nrows=0).columns
esquema = [bigquery.SchemaField(c, 'STRING') for c in columnas]

config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    schema=esquema,
    write_disposition='WRITE_TRUNCATE',
)

with open(RUTA, 'rb') as f:
    job = client.load_table_from_file(f, TABLA, job_config=config)
job.result()

print(f"Filas cargadas: {client.get_table(TABLA).num_rows:,}")

Filas cargadas: 10,000,000
